In [ ]:
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import APIConnectionError, APIStatusError, APITimeoutError, OpenAI, RateLimitError
from tqdm.auto import tqdm

load_dotenv()


# ============================================================================
# 09. LLM REASONING ABLATION (MASKED + FULL TEXT)
# ============================================================================
# 05 and 08 force a structured JSON response with no room to reason before
# answering (max_completion_tokens=80). Every LLM-NLI result in the
# literature this project draws on (Uluslu & Schneider 2025; Zhang & Salle
# 2023) lets the model produce free-text linguistic analysis BEFORE giving a
# label. This notebook removes the JSON constraint and asks the model to
# reason first, then state its answer on a fixed final line parsed with a
# regex. This matters because the near-chance LLM accuracy obtained under the
# forced-JSON format (14.1%, against a 12.5% random baseline for 8 classes)
# may be a prompt-format artifact rather than a genuine capability limit.
#
# Runs on the SAME eval_subset.csv rows as 05/08 for BOTH text conditions
# (masked, fulltext) so all four LLM conditions (05, 08, and this notebook's
# two runs) are directly comparable in 10.
#
# COST ESTIMATION: TOEFL11 essays are short, exam-timed (~350 words on
# average). ICLEv3 essays are untimed and can run several times longer.
# A small random dry-run sample can land disproportionately on one
# partition by chance and badly misestimate the true average cost, so this
# notebook forces the dry run to sample evenly from both partitions and
# reports token usage (and the resulting cost estimate) per partition
# instead of blending them into a single number.

BASE_DIR = Path("..").resolve()
COMBINED_CSV_PATH = BASE_DIR / "data/processed/combined_nli_masked.csv"
EVAL_SUBSET_PATH = BASE_DIR / "data/processed/results/llm_eval_subset.csv"
RESULTS_DIR = BASE_DIR / "data/processed/results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

VALID_L1S = ["DEU", "FRA", "ITA", "JPN", "KOR", "SPA", "TUR", "ZHO"]
MODEL_NAME = "gpt-4o-mini"
MAX_COMPLETION_TOKENS = 450
DRY_RUN_PER_PARTITION = 8  # calls sampled from EACH eval_partition before pausing to report cost
RANDOM_STATE = 42
RUN_FULL_EVALUATION = True  # flip to True per condition after checking the dry-run cost estimate
MAX_RETRIES = 5
# Upper-bound estimate: prices every token at the (higher) output rate ($0.60 / 1M for gpt-4o-mini).
PRICE_PER_MILLION_TOKENS_ESTIMATE = 0.60

TEXT_CONDITIONS = {
    "masked": "masked_text",
    "fulltext": "text",
}

FINAL_ANSWER_RE = re.compile(r"FINAL_ANSWER\s*:\s*(" + "|".join(VALID_L1S) + r")", re.IGNORECASE)

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY is not set in the environment.")

client = OpenAI(api_key=api_key)

BOUNDARY_LEGEND = (
    "The text may contain <P> and <S> tokens marking paragraph and sentence boundaries; "
    "treat them as structural formatting markers, not content."
)

POS_LEGEND = """The essay's content words have been replaced with these placeholders:
- N = noun
- P = proper noun
- V = verb
- A = adjective
- R = adverb
Function words, punctuation, pronouns, determiners, auxiliaries, conjunctions, and
prepositions remain in their original form."""


def build_system_prompt(condition: str) -> str:
    intro = (
        "You are an expert computational linguist and forensic linguist performing Native "
        "Language Identification (NLI) on English essays written by non-native speakers."
    )
    legend = f"\n\n{BOUNDARY_LEGEND}"
    if condition == "masked":
        legend += f"\n\n{POS_LEGEND}"
    instructions = f"""

Think step by step. Identify concrete linguistic clues -- word order, article/preposition use,
function-word patterns, syntactic structure, collocations, or spelling/grammar patterns -- that
point to a specific native language. Write out your reasoning in a short paragraph.

After your reasoning, end your response with exactly one line in this exact format, with no
extra text after it:
FINAL_ANSWER: <CODE>

<CODE> must be exactly one of: {", ".join(VALID_L1S)}."""
    return intro + legend + instructions


def parse_final_answer(raw_text: str):
    match = FINAL_ANSWER_RE.search(raw_text or "")
    return match.group(1).upper() if match else None


def call_llm(system_prompt: str, text: str) -> dict:
    user_prompt = f"Essay:\n\n{text}"

    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0,
                max_completion_tokens=MAX_COMPLETION_TOKENS,
            )

            raw_content = response.choices[0].message.content or ""
            predicted = parse_final_answer(raw_content)
            usage = response.usage

            return {
                "predicted_l1": predicted,
                "raw_response": raw_content,
                "parse_error": predicted is None,
                "prompt_tokens": usage.prompt_tokens if usage else None,
                "completion_tokens": usage.completion_tokens if usage else None,
                "total_tokens": usage.total_tokens if usage else None,
                "error": "",
            }

        except (RateLimitError, APITimeoutError, APIConnectionError, APIStatusError) as e:
            wait_seconds = min(60, 2**attempt)
            if attempt == MAX_RETRIES - 1:
                return {
                    "predicted_l1": None,
                    "raw_response": "",
                    "parse_error": True,
                    "prompt_tokens": None,
                    "completion_tokens": None,
                    "total_tokens": None,
                    "error": repr(e),
                }
            time.sleep(wait_seconds)


def load_eval_subset_with_fulltext() -> pd.DataFrame:
    if not EVAL_SUBSET_PATH.exists():
        raise FileNotFoundError(f"{EVAL_SUBSET_PATH} not found. Run 05_llm_evaluation.ipynb first.")
    eval_subset = pd.read_csv(EVAL_SUBSET_PATH)
    combined = pd.read_csv(COMBINED_CSV_PATH)[["doc_id", "text"]]
    combined["text"] = combined["text"].fillna("").astype(str)
    merged = eval_subset.merge(combined, on="doc_id", how="left", validate="one_to_one")
    if merged["text"].isna().any():
        raise ValueError("Some eval subset rows have no matching unmasked text.")
    return merged


def build_processing_order(df: pd.DataFrame, dry_run_per_partition: int, random_state: int) -> pd.DataFrame:
    """Reorder rows so the dry-run sample draws evenly from every eval_partition
    (TOEFL11, ICLE), instead of relying on whatever partition happens to appear
    first in the already-shuffled row order. Without this, a small dry run could
    by chance include mostly short TOEFL11 essays or mostly long ICLE essays and
    badly misestimate the true average cost."""
    dry_run_parts, remainder_parts = [], []
    for _, part_df in df.groupby("eval_partition", sort=False):
        dry_run_parts.append(part_df.head(dry_run_per_partition))
        remainder_parts.append(part_df.iloc[dry_run_per_partition:])

    dry_run_rows = pd.concat(dry_run_parts).sample(frac=1, random_state=random_state)
    remainder_rows = pd.concat(remainder_parts).sample(frac=1, random_state=random_state)
    return pd.concat([dry_run_rows, remainder_rows], ignore_index=False)


def summarize_dry_run(progress_df: pd.DataFrame, eval_df: pd.DataFrame, price_per_million_tokens: float) -> None:
    """Print a per-partition token/cost estimate instead of one blended mean.
    See the notebook header for why blending TOEFL11 and ICLE token counts
    together can be misleading."""
    dry_run_df = progress_df[progress_df["total_tokens"].notna()]
    partition_sizes = eval_df["eval_partition"].value_counts()

    print("Dry run complete. Token usage by partition:")
    weighted_total_tokens = 0.0
    for partition, part_dry in dry_run_df.groupby("eval_partition"):
        mean_tokens = part_dry["total_tokens"].mean()
        n_full = int(partition_sizes.get(partition, 0))
        partition_estimate = mean_tokens * n_full
        weighted_total_tokens += partition_estimate
        print(
            f"  {partition}: mean {mean_tokens:.1f} tokens/call "
            f"(from {len(part_dry)} sampled calls) -> ~{partition_estimate:.0f} tokens across all {n_full} rows"
        )

    missing_partitions = set(partition_sizes.index) - set(dry_run_df["eval_partition"].unique())
    if missing_partitions:
        print(f"  WARNING: no dry-run calls yet for {sorted(missing_partitions)} -- estimate below is incomplete.")

    estimated_cost_usd = weighted_total_tokens / 1_000_000 * price_per_million_tokens
    parse_failures = dry_run_df["parse_error"].sum()
    print(f"Weighted total estimated tokens (all partitions): {weighted_total_tokens:.0f}")
    print(f"Rough estimated cost: ${estimated_cost_usd:.3f}")
    print(f"Parse failures in dry run: {parse_failures} / {len(dry_run_df)}")
    print("Inspect cost and parse-failure rate before continuing.")


eval_df = load_eval_subset_with_fulltext()
eval_df = build_processing_order(eval_df, DRY_RUN_PER_PARTITION, RANDOM_STATE)
n_partitions = eval_df["eval_partition"].nunique()
DRY_RUN_N = DRY_RUN_PER_PARTITION * n_partitions
print(f"Dry run will sample {DRY_RUN_PER_PARTITION} calls from each of {n_partitions} partitions ({DRY_RUN_N} total) per condition.")

for condition_name, text_column in TEXT_CONDITIONS.items():
    print(f"\n{'=' * 80}\nCondition: {condition_name} (column: {text_column})\n{'=' * 80}")

    output_path = RESULTS_DIR / f"llm_predictions_{condition_name}_reasoning.csv"
    progress_path = RESULTS_DIR / f"llm_predictions_{condition_name}_reasoning_progress.csv"
    system_prompt = build_system_prompt(condition_name)

    existing = pd.DataFrame()
    completed_ids = set()
    if progress_path.exists():
        existing = pd.read_csv(progress_path)
        completed_ids = set(existing["eval_id"])
        print(f"Resuming from progress file. Completed rows: {len(completed_ids)}")

    rows = [] if existing.empty else existing.to_dict("records")
    new_calls_made = 0

    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
        if row["eval_id"] in completed_ids:
            continue

        result = call_llm(system_prompt, row[text_column])
        output_row = {
            "eval_id": row["eval_id"],
            "doc_id": row["doc_id"],
            "dataset": row["dataset"],
            "split": row["split"],
            "eval_partition": row["eval_partition"],
            "true_l1": row["label"],
            "model": MODEL_NAME,
            "text_condition": condition_name,
            "prompt_format": "reasoning",
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            **result,
        }
        rows.append(output_row)
        new_calls_made += 1

        pd.DataFrame(rows).to_csv(progress_path, index=False)

        if new_calls_made == DRY_RUN_N:
            progress_df = pd.DataFrame(rows)
            summarize_dry_run(progress_df, eval_df, PRICE_PER_MILLION_TOKENS_ESTIMATE)
            print(f"To run all remaining rows for '{condition_name}', set RUN_FULL_EVALUATION = True and rerun.")
            if not RUN_FULL_EVALUATION:
                break

    results_df = pd.DataFrame(rows)
    results_df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")
    if len(results_df):
        print(f"Parse failure rate: {results_df['parse_error'].mean():.2%}")
